In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are now expected on the local filesystem. Set LOCAL_DATA_DIR
# (or rely on the notebook's working directory) to point to the folder that
# contains data.csv, predict.csv, etc.

import os
from pathlib import Path

LOCAL_DATA_DIR = Path(os.environ.get("LOCAL_DATA_DIR", Path.cwd())).resolve() / "data"
print(f"LOCAL_DATA_DIR set to {LOCAL_DATA_DIR}")

LOCAL_DATA_DIR set to /home/fredc/NTU-DSAI/kaggle-ha/data


In [3]:
## Starter Code for Building Baseball Win Prediction Model

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Load the pre-processed datasets from the directory pointed to by LOCAL_DATA_DIR
DATA_DIR = LOCAL_DATA_DIR  # defined in the previous cell
print(f"Loading datasets from {DATA_DIR}")

data_df = pd.read_csv(DATA_DIR / "data.csv")
predict_df = pd.read_csv(DATA_DIR / "predict.csv")

# Display basic information about the datasets
print(f"Data set shape: {data_df.shape}")
print(f"Predict set shape: {predict_df.shape}")

Loading datasets from /home/fredc/NTU-DSAI/kaggle-ha/data
Data set shape: (1812, 51)
Predict set shape: (453, 45)


In [4]:
def add_engineered_features(df):
    d = df.copy()
    G  = d['G']
    IP = d['IPouts'] / 3
    PA = d['AB'] + d['BB']

    d['run_diff']        = d['R'] - d['RA']
    d['run_diff_pg']     = d['run_diff'] / G
    # Fixed 1.83 exponent: baseball-research optimal for MLB
    d['pythag_wp']       = d['R']**1.83 / (d['R']**1.83 + d['RA']**1.83 + 1e-9)
    d['pyth_wins']       = d['pythag_wp'] * G
    # PythagenPat: dynamic exponent = ((R+RA)/G)^0.287
    dyn_exp = ((d['R'] + d['RA']) / (G + 1e-9)) ** 0.287
    d['pythagenport_wp'] = d['R']**dyn_exp / (d['R']**dyn_exp + d['RA']**dyn_exp + 1e-9)
    d['pythagenport_wins'] = d['pythagenport_wp'] * G

    d['batting_avg']     = d['H'] / d['AB']
    d['obp_proxy']       = (d['H'] + d['BB']) / PA
    # Correct SLG: singles counted explicitly
    singles              = d['H'] - d['2B'] - d['3B'] - d['HR']
    d['slg_proxy']       = (singles + 2*d['2B'] + 3*d['3B'] + 4*d['HR']) / (d['AB'] + 1e-9)
    d['ops_proxy']       = d['obp_proxy'] + d['slg_proxy']
    d['iso']             = d['slg_proxy'] - d['batting_avg']  # isolated power
    d['hr_rate_off']     = d['HR'] / d['AB']
    d['bb_rate']         = d['BB'] / PA
    d['so_rate']         = d['SO'] / PA

    d['whip']            = (d['BBA'] + d['HA']) / IP
    d['k_bb_ratio']      = d['SOA'] / d['BBA'].replace(0, float('nan'))
    d['hr_rate_def']     = d['HRA'] / IP
    d['fip_proxy']       = (13*d['HRA'] + 3*d['BBA'] - 2*d['SOA']) / IP + 3.2
    d['era_vs_league']   = d['ERA'] / d['mlb_rpg'].replace(0, float('nan'))

    d['sv_rate']         = d['SV'] / G
    d['cg_rate']         = d['CG'] / G
    d['sho_rate']        = d['SHO'] / G
    d['r_vs_lg']         = d['R'] / G - d['mlb_rpg']
    d['ra_vs_lg']        = d['RA'] / G - d['mlb_rpg']

    return d.fillna(0)

data_df    = add_engineered_features(data_df)
predict_df = add_engineered_features(predict_df)

eng_cols = [
    'run_diff','run_diff_pg',
    'pythag_wp','pyth_wins','pythagenport_wp','pythagenport_wins',
    'batting_avg','obp_proxy','slg_proxy','ops_proxy','iso',
    'hr_rate_off','bb_rate','so_rate',
    'whip','k_bb_ratio','hr_rate_def','fip_proxy','era_vs_league',
    'sv_rate','cg_rate','sho_rate','r_vs_lg','ra_vs_lg'
]
print(f"Engineered {len(eng_cols)} new features.")


Engineered 24 new features.


In [5]:
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split, KFold, cross_val_score

base_features = [
    'G', 'R', 'AB', 'H', '2B', '3B', 'HR', 'BB', 'SO', 'SB',
    'RA', 'ER', 'ERA', 'CG', 'SHO', 'SV', 'IPouts', 'HA', 'HRA', 'BBA', 'SOA',
    'E', 'DP', 'FP', 'mlb_rpg',
    'era_1', 'era_2', 'era_3', 'era_4', 'era_5', 'era_6', 'era_7', 'era_8',
    'decade_1910', 'decade_1920', 'decade_1930', 'decade_1940', 'decade_1950',
    'decade_1960', 'decade_1970', 'decade_1980', 'decade_1990', 'decade_2000', 'decade_2010',
] + eng_cols

available_features = [c for c in base_features if c in data_df.columns and c in predict_df.columns]
print(f"Total features: {len(available_features)}")

X = data_df[available_features]
y = data_df['W']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")


Total features: 68
Train: 1449  |  Test: 363


In [6]:
from sklearn.preprocessing import StandardScaler

one_hot_cols = [col for col in X_train.columns if col.startswith(('era_', 'decade_'))]
other_cols   = [col for col in X_train.columns if col not in one_hot_cols]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[other_cols] = scaler.fit_transform(X_train[other_cols])
X_test_scaled[other_cols]  = scaler.transform(X_test[other_cols])
print(f"Scaling {len(other_cols)} continuous features; {len(one_hot_cols)} binary cols left unscaled.")


Scaling 48 continuous features; 20 binary cols left unscaled.


In [7]:
alphas = np.logspace(-3, 5, 100)

ridge = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_absolute_error')
ridge.fit(X_train_scaled, y_train)
print(f"Best alpha: {ridge.alpha_:.4f}")


Best alpha: 0.9770


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

train_preds = ridge.predict(X_train_scaled)
test_preds  = ridge.predict(X_test_scaled)

print("Ridge Performance (80/20 split):")
print(f"  Train MAE:  {mean_absolute_error(y_train, train_preds):.4f}")
print(f"  Test  MAE:  {mean_absolute_error(y_test, test_preds):.4f}")
print(f"  Test  RMSE: {mean_squared_error(y_test, test_preds)**0.5:.4f}")
print(f"  Test  R2:   {r2_score(y_test, test_preds):.4f}")

# 10-fold CV on full data — more reliable estimate
X_all_scaled = X.copy()
X_all_scaled[other_cols] = scaler.transform(X[other_cols])

kf = KFold(n_splits=10, shuffle=True, random_state=0)
cv_scores = cross_val_score(
    RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_absolute_error'),
    X_all_scaled, y,
    cv=kf, scoring='neg_mean_absolute_error'
)
cv_mae = -cv_scores.mean()
print(f"\n10-fold CV MAE (full data): {cv_mae:.4f} +/- {cv_scores.std():.4f}")
print(f"Baseline LR public score:   3.0800")
print(f"Est. public (CV + ~0.33 gap): {cv_mae + 0.33:.4f}")


Ridge Performance (80/20 split):
  Train MAE:  2.6120
  Test  MAE:  2.8057
  Test  RMSE: 3.5274
  Test  R2:   0.9225

10-fold CV MAE (full data): 2.7126 +/- 0.1565
Baseline LR public score:   3.0800
Est. public (CV + ~0.33 gap): 3.0426


In [9]:
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.model_selection import KFold
import itertools

# --- ElasticNet tuning grid ---
# l1_ratio=0 is pure Ridge, l1_ratio=1 is pure Lasso
l1_ratios = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
alphas_en  = np.logspace(-4, 2, 60)

kf_en = KFold(n_splits=5, shuffle=True, random_state=42)

enet_cv = ElasticNetCV(
    l1_ratio=l1_ratios,
    alphas=alphas_en,
    cv=kf_en,
    max_iter=50000,
    tol=1e-4,
    n_jobs=-1,
)
enet_cv.fit(X_train_scaled, y_train)

print(f"Best alpha:    {enet_cv.alpha_:.6f}")
print(f"Best l1_ratio: {enet_cv.l1_ratio_:.4f}")

# Evaluate on 80/20 split
en_train_preds = enet_cv.predict(X_train_scaled)
en_test_preds  = enet_cv.predict(X_test_scaled)

en_test_mae  = mean_absolute_error(y_test, en_test_preds)
en_test_rmse = mean_squared_error(y_test, en_test_preds) ** 0.5
en_test_r2   = r2_score(y_test, en_test_preds)

print(f"\nElasticNet Performance (80/20 split):")
print(f"  Train MAE:  {mean_absolute_error(y_train, en_train_preds):.4f}")
print(f"  Test  MAE:  {en_test_mae:.4f}")
print(f"  Test  RMSE: {en_test_rmse:.4f}")
print(f"  Test  R2:   {en_test_r2:.4f}")

# Count zeroed-out features
n_zero = (enet_cv.coef_ == 0).sum()
n_total = len(enet_cv.coef_)
print(f"\nFeature selection: {n_zero}/{n_total} coefficients zeroed out")
print(f"Active features:  {n_total - n_zero}")

# 10-fold CV on full data for reliable estimate
enet_full = ElasticNet(alpha=enet_cv.alpha_, l1_ratio=enet_cv.l1_ratio_, max_iter=50000)
kf_full = KFold(n_splits=10, shuffle=True, random_state=0)
en_cv_scores = cross_val_score(enet_full, X_all_scaled, y, cv=kf_full, scoring='neg_mean_absolute_error')
en_cv_mae = -en_cv_scores.mean()
print(f"\n10-fold CV MAE (full data): {en_cv_mae:.4f} +/- {en_cv_scores.std():.4f}")

# Compare with Ridge
print(f"\n--- Comparison ---")
print(f"  Ridge      test MAE: {mean_absolute_error(y_test, test_preds):.4f}")
print(f"  ElasticNet test MAE: {en_test_mae:.4f}")
print(f"  Improvement:         {mean_absolute_error(y_test, test_preds) - en_test_mae:+.4f}")


/home/fredc/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.221e+02, tolerance: 2.041e+01
  model = cd_fast.enet_coordinate_descent_gram(
/home/fredc/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.856e+01, tolerance: 2.048e+01
  model = cd_fast.enet_coordinate_descent_gram(
/home/fredc/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or conside

Best alpha:    0.005356
Best l1_ratio: 0.9900

ElasticNet Performance (80/20 split):
  Train MAE:  2.6222
  Test  MAE:  2.8015
  Test  RMSE: 3.5242
  Test  R2:   0.9226

Feature selection: 32/68 coefficients zeroed out
Active features:  36

10-fold CV MAE (full data): 2.7055 +/- 0.1566

--- Comparison ---
  Ridge      test MAE: 2.8057
  ElasticNet test MAE: 2.8015
  Improvement:         +0.0043


---
### Prerequisite checkpoint
**Run cells 0–7 above** before executing the win-percentage target transformation cell at the bottom of this notebook (cell 22+). Those cells provide: `data_df`, `predict_df`, feature engineering, `X`/`y` splits, scaling, Ridge, and ElasticNet baselines.

> **Prerequisite checkpoint**
> 
> Run **cells 0–7 above** before executing the win-percentage target transformation cell at the bottom of this notebook. Those cells provide: `data_df`, `predict_df`, feature engineering, `X`/`y` splits, scaling, Ridge, and ElasticNet baselines.

In [28]:
from sklearn.linear_model import HuberRegressor
from sklearn.model_selection import GridSearchCV

# --- Identify ElasticNet-selected features ---
active_mask = enet_cv.coef_ != 0
active_features = [f for f, m in zip(available_features, active_mask) if m]
print(f"ElasticNet active features: {len(active_features)}/{len(available_features)}")
print(f"  {active_features}")

# --- Huber tuning grid ---
huber_params = {
    'epsilon': [1.1, 1.2, 1.35, 1.5, 1.75, 2.0, 2.5, 3.0],
    'alpha':   [1e-4, 5e-4, 1e-3, 5e-3, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
}
print(f"Grid: {len(huber_params['epsilon'])} x {len(huber_params['alpha'])} = "
      f"{len(huber_params['epsilon']) * len(huber_params['alpha'])} combos")

kf_hub = KFold(n_splits=5, shuffle=True, random_state=42)

# --- Variant 1: Huber on ALL 68 features (L2 only) ---
hub_all = GridSearchCV(
    HuberRegressor(max_iter=50000),
    huber_params,
    cv=kf_hub,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
hub_all.fit(X_train_scaled, y_train)

hub_all_test_preds = hub_all.predict(X_test_scaled)
hub_all_mae = mean_absolute_error(y_test, hub_all_test_preds)

print(f"\n=== Huber (all {len(available_features)} features) ===")
print(f"  Best epsilon: {hub_all.best_params_['epsilon']}")
print(f"  Best alpha:   {hub_all.best_params_['alpha']}")
print(f"  Train MAE:    {mean_absolute_error(y_train, hub_all.predict(X_train_scaled)):.4f}")
print(f"  Test  MAE:    {hub_all_mae:.4f}")
print(f"  Test  RMSE:   {mean_squared_error(y_test, hub_all_test_preds)**0.5:.4f}")
print(f"  Test  R2:     {r2_score(y_test, hub_all_test_preds):.4f}")

# --- Variant 2: Huber on ElasticNet-selected features only ---
active_other = [c for c in active_features if c not in one_hot_cols]

X_train_active = X_train_scaled[active_features]
X_test_active  = X_test_scaled[active_features]

hub_sel = GridSearchCV(
    HuberRegressor(max_iter=50000),
    huber_params,
    cv=kf_hub,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
hub_sel.fit(X_train_active, y_train)

hub_sel_test_preds = hub_sel.predict(X_test_active)
hub_sel_mae = mean_absolute_error(y_test, hub_sel_test_preds)

print(f"\n=== Huber (ElasticNet-selected {len(active_features)} features) ===")
print(f"  Best epsilon: {hub_sel.best_params_['epsilon']}")
print(f"  Best alpha:   {hub_sel.best_params_['alpha']}")
print(f"  Train MAE:    {mean_absolute_error(y_train, hub_sel.predict(X_train_active)):.4f}")
print(f"  Test  MAE:    {hub_sel_mae:.4f}")
print(f"  Test  RMSE:   {mean_squared_error(y_test, hub_sel_test_preds)**0.5:.4f}")
print(f"  Test  R2:     {r2_score(y_test, hub_sel_test_preds):.4f}")

# --- 10-fold CV on full data for both ---
X_all_active = X_all_scaled[active_features]

kf_full = KFold(n_splits=10, shuffle=True, random_state=0)

hub_all_cv = cross_val_score(
    HuberRegressor(epsilon=hub_all.best_params_['epsilon'],
                   alpha=hub_all.best_params_['alpha'], max_iter=50000),
    X_all_scaled, y, cv=kf_full, scoring='neg_mean_absolute_error'
)
hub_sel_cv = cross_val_score(
    HuberRegressor(epsilon=hub_sel.best_params_['epsilon'],
                   alpha=hub_sel.best_params_['alpha'], max_iter=50000),
    X_all_active, y, cv=kf_full, scoring='neg_mean_absolute_error'
)

print(f"\n=== 10-fold CV (full data) ===")
print(f"  Huber (all feat)  CV MAE: {-hub_all_cv.mean():.4f} +/- {hub_all_cv.std():.4f}")
print(f"  Huber (selected)  CV MAE: {-hub_sel_cv.mean():.4f} +/- {hub_sel_cv.std():.4f}")

# --- Summary ---
print(f"\n{'='*50}")
print(f"  Model               Test MAE   10-fold CV MAE")
print(f"  Ridge               {mean_absolute_error(y_test, test_preds):.4f}     {cv_mae:.4f}")
print(f"  ElasticNet          {en_test_mae:.4f}     {en_cv_mae:.4f}")
print(f"  Huber (all feat)    {hub_all_mae:.4f}     {-hub_all_cv.mean():.4f}")
print(f"  Huber (selected)    {hub_sel_mae:.4f}     {-hub_sel_cv.mean():.4f}")
print(f"{'='*50}")

# Pick best Huber variant
best_hub_name, best_hub_mae, best_hub_model, best_hub_features = (
    ("Huber_all", hub_all_mae, hub_all, available_features)
    if hub_all_mae <= hub_sel_mae
    else ("Huber_selected", hub_sel_mae, hub_sel, active_features)
)
print(f"\nBest Huber variant: {best_hub_name} (test MAE={best_hub_mae:.4f})")


ElasticNet active features: 36/68
  ['G', 'R', 'AB', '2B', '3B', 'SB', 'CG', 'SHO', 'SV', 'IPouts', 'HA', 'E', 'DP', 'mlb_rpg', 'era_1', 'era_2', 'era_3', 'era_4', 'era_6', 'era_7', 'decade_1910', 'decade_1920', 'decade_1930', 'decade_1940', 'decade_1970', 'decade_1980', 'decade_1990', 'pythagenport_wp', 'pythagenport_wins', 'batting_avg', 'slg_proxy', 'bb_rate', 'whip', 'k_bb_ratio', 'fip_proxy', 'cg_rate']
Grid: 8 x 10 = 80 combos

=== Huber (all 68 features) ===
  Best epsilon: 3.0
  Best alpha:   0.5
  Train MAE:    2.6146
  Test  MAE:    2.8028
  Test  RMSE:   3.5256
  Test  R2:     0.9226

=== Huber (ElasticNet-selected 36 features) ===
  Best epsilon: 3.0
  Best alpha:   0.5
  Train MAE:    2.6170
  Test  MAE:    2.7991
  Test  RMSE:   3.5269
  Test  R2:     0.9225

=== 10-fold CV (full data) ===
  Huber (all feat)  CV MAE: 2.7096 +/- 0.1527
  Huber (selected)  CV MAE: 2.6954 +/- 0.1516

  Model               Test MAE   10-fold CV MAE
  Ridge               2.8057     2.7126
  El

In [29]:
# --- Conditional submission: deploy best Huber if test MAE < 2.81 ---
if best_hub_mae < 2.81:
    print(f"{best_hub_name} test MAE ({best_hub_mae:.4f}) < 2.81 — generating submission.")

    best_params = best_hub_model.best_params_

    # Refit scaler + model on ALL training data
    scaler_hub = StandardScaler()
    hub_other = [c for c in best_hub_features if c not in one_hot_cols]

    X_all_hub = X[best_hub_features].copy()
    X_all_hub[hub_other] = scaler_hub.fit_transform(X_all_hub[hub_other])

    final_hub = HuberRegressor(
        epsilon=best_params['epsilon'],
        alpha=best_params['alpha'],
        max_iter=50000,
    )
    final_hub.fit(X_all_hub, y)

    predict_hub = predict_df[best_hub_features].copy()
    predict_hub[hub_other] = scaler_hub.transform(predict_hub[hub_other])
    hub_preds = final_hub.predict(predict_hub)

    print(f"Predictions: min={hub_preds.min():.1f}  mean={hub_preds.mean():.1f}  max={hub_preds.max():.1f}")

    from datetime import datetime
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"submission_{best_hub_name}_{ts}.csv"
    pd.DataFrame({"ID": predict_df["ID"], "W": hub_preds.round().astype(int)}).to_csv(filename, index=False)
    print(f"Saved: {filename}")

    # Submit to Kaggle
    msg = f"{best_hub_name} eps={best_params['epsilon']} alpha={best_params['alpha']} testMAE={best_hub_mae:.4f}"
    cmd = f'kaggle competitions submit -c {COMPETITION_NAME} -f "{filename}" -m "{msg}"'
    print(f"Submitting: {cmd}")
    os.system(cmd)
else:
    print(f"{best_hub_name} test MAE ({best_hub_mae:.4f}) >= 2.81 — skipping submission.")
    print("ElasticNet remains the best submitted model.")


Huber_selected test MAE (2.7991) < 2.81 — generating submission.
Predictions: min=44.9  mean=79.0  max=109.6
Saved: submission_Huber_selected_20260329_182330.csv
Submitting: kaggle competitions submit -c sctpdsai-m-3-ds-2-f-coaching-money-ball-analytics -f "submission_Huber_selected_20260329_182330.csv" -m "Huber_selected eps=3.0 alpha=0.5 testMAE=2.7991"


100%|██████████| 3.34k/3.34k [00:00<00:00, 4.41kB/s]


Successfully submitted to SCTPDSAI-M3-DS2F-Coaching-MoneyBall Analytics

In [31]:
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import GridSearchCV, KFold

# --- Poisson GLM on ElasticNet-selected features (36) ---
# PoissonRegressor uses log-link: predictions = exp(X @ coef), always non-negative
# Only L2 penalty available, so we rely on ElasticNet's prior L1 selection

# Poisson needs non-negative target (wins are counts ≥ 0) ✓
print(f"Target range: {y.min()} – {y.max()} (all non-negative: {(y >= 0).all()})")
print(f"Using {len(active_features)} ElasticNet-selected features")

poisson_params = {
    'alpha': [1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
}

kf_poi = KFold(n_splits=5, shuffle=True, random_state=42)

# Variant 1: Poisson on ElasticNet-selected features
X_train_active = X_train_scaled[active_features]
X_test_active  = X_test_scaled[active_features]

poi_sel = GridSearchCV(
    PoissonRegressor(max_iter=10000),
    poisson_params,
    cv=kf_poi,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
poi_sel.fit(X_train_active, y_train)

poi_sel_test_preds = poi_sel.predict(X_test_active)
poi_sel_mae = mean_absolute_error(y_test, poi_sel_test_preds)

print(f"\n=== Poisson (ElasticNet-selected {len(active_features)} features) ===")
print(f"  Best alpha:   {poi_sel.best_params_['alpha']}")
print(f"  Train MAE:    {mean_absolute_error(y_train, poi_sel.predict(X_train_active)):.4f}")
print(f"  Test  MAE:    {poi_sel_mae:.4f}")
print(f"  Test  RMSE:   {mean_squared_error(y_test, poi_sel_test_preds)**0.5:.4f}")
print(f"  Test  R2:     {r2_score(y_test, poi_sel_test_preds):.4f}")

# Variant 2: Poisson on ALL features (for comparison)
poi_all = GridSearchCV(
    PoissonRegressor(max_iter=10000),
    poisson_params,
    cv=kf_poi,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
poi_all.fit(X_train_scaled, y_train)

poi_all_test_preds = poi_all.predict(X_test_scaled)
poi_all_mae = mean_absolute_error(y_test, poi_all_test_preds)

print(f"\n=== Poisson (all {len(available_features)} features) ===")
print(f"  Best alpha:   {poi_all.best_params_['alpha']}")
print(f"  Train MAE:    {mean_absolute_error(y_train, poi_all.predict(X_train_scaled)):.4f}")
print(f"  Test  MAE:    {poi_all_mae:.4f}")
print(f"  Test  RMSE:   {mean_squared_error(y_test, poi_all_test_preds)**0.5:.4f}")
print(f"  Test  R2:     {r2_score(y_test, poi_all_test_preds):.4f}")

# 10-fold CV on full data
X_all_active = X_all_scaled[active_features]
kf_full = KFold(n_splits=10, shuffle=True, random_state=0)

poi_sel_cv = cross_val_score(
    PoissonRegressor(alpha=poi_sel.best_params_['alpha'], max_iter=10000),
    X_all_active, y, cv=kf_full, scoring='neg_mean_absolute_error'
)
poi_all_cv = cross_val_score(
    PoissonRegressor(alpha=poi_all.best_params_['alpha'], max_iter=10000),
    X_all_scaled, y, cv=kf_full, scoring='neg_mean_absolute_error'
)

print(f"\n=== 10-fold CV (full data) ===")
print(f"  Poisson (selected) CV MAE: {-poi_sel_cv.mean():.4f} +/- {poi_sel_cv.std():.4f}")
print(f"  Poisson (all feat) CV MAE: {-poi_all_cv.mean():.4f} +/- {poi_all_cv.std():.4f}")

# Summary
print(f"\n{'='*55}")
print(f"  Model                  Test MAE   10-fold CV MAE")
print(f"  Ridge                  {mean_absolute_error(y_test, test_preds):.4f}     {cv_mae:.4f}")
print(f"  ElasticNet             {en_test_mae:.4f}     {en_cv_mae:.4f}")
print(f"  Poisson (selected)     {poi_sel_mae:.4f}     {-poi_sel_cv.mean():.4f}")
print(f"  Poisson (all feat)     {poi_all_mae:.4f}     {-poi_all_cv.mean():.4f}")
print(f"{'='*55}")

# Pick best Poisson variant
best_poi_name, best_poi_mae, best_poi_model, best_poi_features = (
    ("Poisson_all", poi_all_mae, poi_all, available_features)
    if poi_all_mae <= poi_sel_mae
    else ("Poisson_selected", poi_sel_mae, poi_sel, active_features)
)
print(f"\nBest Poisson variant: {best_poi_name} (test MAE={best_poi_mae:.4f})")


Target range: 36 – 116 (all non-negative: True)
Using 36 ElasticNet-selected features

=== Poisson (ElasticNet-selected 36 features) ===
  Best alpha:   5e-05
  Train MAE:    2.8326
  Test  MAE:    3.0260
  Test  RMSE:   3.8373
  Test  R2:     0.9083

=== Poisson (all 68 features) ===
  Best alpha:   0.001
  Train MAE:    2.7677
  Test  MAE:    3.0248
  Test  RMSE:   3.8057
  Test  R2:     0.9098

=== 10-fold CV (full data) ===
  Poisson (selected) CV MAE: 2.9174 +/- 0.1376
  Poisson (all feat) CV MAE: 2.8904 +/- 0.1357

  Model                  Test MAE   10-fold CV MAE
  Ridge                  2.8057     2.7126
  ElasticNet             2.8015     2.7055
  Poisson (selected)     3.0260     2.9174
  Poisson (all feat)     3.0248     2.8904

Best Poisson variant: Poisson_all (test MAE=3.0248)


In [32]:
# --- Conditional submission: deploy best Poisson if test MAE < 2.81 ---
if best_poi_mae < 2.81:
    print(f"{best_poi_name} test MAE ({best_poi_mae:.4f}) < 2.81 — generating submission.")

    best_poi_params = best_poi_model.best_params_
    poi_other = [c for c in best_poi_features if c not in one_hot_cols]

    # Refit scaler + model on ALL training data
    scaler_poi = StandardScaler()
    X_all_poi = X[best_poi_features].copy()
    X_all_poi[poi_other] = scaler_poi.fit_transform(X_all_poi[poi_other])

    final_poi = PoissonRegressor(
        alpha=best_poi_params['alpha'],
        max_iter=10000,
    )
    final_poi.fit(X_all_poi, y)

    predict_poi = predict_df[best_poi_features].copy()
    predict_poi[poi_other] = scaler_poi.transform(predict_poi[poi_other])
    poi_preds = final_poi.predict(predict_poi)

    print(f"Predictions: min={poi_preds.min():.1f}  mean={poi_preds.mean():.1f}  max={poi_preds.max():.1f}")

    from datetime import datetime
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"submission_{best_poi_name}_{ts}.csv"
    pd.DataFrame({"ID": predict_df["ID"], "W": poi_preds.round().astype(int)}).to_csv(filename, index=False)
    print(f"Saved: {filename}")

    msg = f"{best_poi_name} alpha={best_poi_params['alpha']} testMAE={best_poi_mae:.4f}"
    cmd = f'kaggle competitions submit -c {COMPETITION_NAME} -f "{filename}" -m "{msg}"'
    print(f"Submitting: {cmd}")
    os.system(cmd)
else:
    print(f"{best_poi_name} test MAE ({best_poi_mae:.4f}) >= 2.81 — skipping submission.")
    print("ElasticNet remains the best submitted model.")


Poisson_all test MAE (3.0248) >= 2.81 — skipping submission.
ElasticNet remains the best submitted model.


In [34]:
# --- Two-Stage: ElasticNet feature selection → RidgeCV refit ---
from sklearn.linear_model import RidgeCV, Ridge, LinearRegression

print(f"Stage 1: ElasticNet selected {len(active_features)} features (from {len(available_features)})")
print(f"  Features: {active_features}\n")

# Prepare active-only data
X_train_active = X_train_scaled[active_features]
X_test_active  = X_test_scaled[active_features]
X_all_active   = X_all_scaled[active_features]

# --- Stage 2a: RidgeCV on selected features ---
alphas_s2 = np.logspace(-3, 5, 100)
ridge_s2 = RidgeCV(alphas=alphas_s2, cv=5, scoring='neg_mean_absolute_error')
ridge_s2.fit(X_train_active, y_train)

ridge_s2_test_preds = ridge_s2.predict(X_test_active)
ridge_s2_mae = mean_absolute_error(y_test, ridge_s2_test_preds)

print(f"=== Stage 2a: RidgeCV on {len(active_features)} selected features ===")
print(f"  Best alpha:  {ridge_s2.alpha_:.4f}")
print(f"  Train MAE:   {mean_absolute_error(y_train, ridge_s2.predict(X_train_active)):.4f}")
print(f"  Test  MAE:   {ridge_s2_mae:.4f}")
print(f"  Test  RMSE:  {mean_squared_error(y_test, ridge_s2_test_preds)**0.5:.4f}")
print(f"  Test  R2:    {r2_score(y_test, ridge_s2_test_preds):.4f}")

# --- Stage 2b: OLS on selected features (no regularisation) ---
ols_s2 = LinearRegression()
ols_s2.fit(X_train_active, y_train)

ols_s2_test_preds = ols_s2.predict(X_test_active)
ols_s2_mae = mean_absolute_error(y_test, ols_s2_test_preds)

print(f"\n=== Stage 2b: OLS on {len(active_features)} selected features ===")
print(f"  Train MAE:   {mean_absolute_error(y_train, ols_s2.predict(X_train_active)):.4f}")
print(f"  Test  MAE:   {ols_s2_mae:.4f}")
print(f"  Test  RMSE:  {mean_squared_error(y_test, ols_s2_test_preds)**0.5:.4f}")
print(f"  Test  R2:    {r2_score(y_test, ols_s2_test_preds):.4f}")

# --- 10-fold CV on full data ---
kf_full = KFold(n_splits=10, shuffle=True, random_state=0)

ridge_s2_cv = cross_val_score(
    RidgeCV(alphas=alphas_s2, cv=5, scoring='neg_mean_absolute_error'),
    X_all_active, y, cv=kf_full, scoring='neg_mean_absolute_error'
)
ols_s2_cv = cross_val_score(
    LinearRegression(),
    X_all_active, y, cv=kf_full, scoring='neg_mean_absolute_error'
)

print(f"\n=== 10-fold CV (full data) ===")
print(f"  Ridge (selected)  CV MAE: {-ridge_s2_cv.mean():.4f} +/- {ridge_s2_cv.std():.4f}")
print(f"  OLS   (selected)  CV MAE: {-ols_s2_cv.mean():.4f} +/- {ols_s2_cv.std():.4f}")

# --- Summary ---
print(f"\n{'='*60}")
print(f"  Model                       Test MAE   10-fold CV MAE")
print(f"  Ridge (68 feat)             {mean_absolute_error(y_test, test_preds):.4f}     {cv_mae:.4f}")
print(f"  ElasticNet (68→36 feat)     {en_test_mae:.4f}     {en_cv_mae:.4f}")
print(f"  2-stage Ridge (36 feat)     {ridge_s2_mae:.4f}     {-ridge_s2_cv.mean():.4f}")
print(f"  2-stage OLS   (36 feat)     {ols_s2_mae:.4f}     {-ols_s2_cv.mean():.4f}")
print(f"{'='*60}")

# Pick best two-stage variant
if ridge_s2_mae <= ols_s2_mae:
    best_s2_name, best_s2_mae, best_s2_model = "TwoStage_Ridge", ridge_s2_mae, ridge_s2
else:
    best_s2_name, best_s2_mae, best_s2_model = "TwoStage_OLS", ols_s2_mae, ols_s2
print(f"\nBest two-stage variant: {best_s2_name} (test MAE={best_s2_mae:.4f})")


Stage 1: ElasticNet selected 36 features (from 68)
  Features: ['G', 'R', 'AB', '2B', '3B', 'SB', 'CG', 'SHO', 'SV', 'IPouts', 'HA', 'E', 'DP', 'mlb_rpg', 'era_1', 'era_2', 'era_3', 'era_4', 'era_6', 'era_7', 'decade_1910', 'decade_1920', 'decade_1930', 'decade_1940', 'decade_1970', 'decade_1980', 'decade_1990', 'pythagenport_wp', 'pythagenport_wins', 'batting_avg', 'slg_proxy', 'bb_rate', 'whip', 'k_bb_ratio', 'fip_proxy', 'cg_rate']

=== Stage 2a: RidgeCV on 36 selected features ===
  Best alpha:  0.5591
  Train MAE:   2.6153
  Test  MAE:   2.8029
  Test  RMSE:  3.5310
  Test  R2:    0.9223

=== Stage 2b: OLS on 36 selected features ===
  Train MAE:   2.6146
  Test  MAE:   2.8061
  Test  RMSE:  3.5332
  Test  R2:    0.9223

=== 10-fold CV (full data) ===
  Ridge (selected)  CV MAE: 2.6966 +/- 0.1533
  OLS   (selected)  CV MAE: 2.7002 +/- 0.1507

  Model                       Test MAE   10-fold CV MAE
  Ridge (68 feat)             2.8057     2.7126
  ElasticNet (68→36 feat)     2.8015

In [35]:
# --- Conditional submission: deploy best two-stage model if test MAE < 2.81 ---
if best_s2_mae < 2.81:
    print(f"{best_s2_name} test MAE ({best_s2_mae:.4f}) < 2.81 — generating submission.")

    s2_other = [c for c in active_features if c not in one_hot_cols]

    # Refit scaler + model on ALL training data
    scaler_s2 = StandardScaler()
    X_all_s2 = X[active_features].copy()
    X_all_s2[s2_other] = scaler_s2.fit_transform(X_all_s2[s2_other])

    if best_s2_name == "TwoStage_Ridge":
        final_s2 = Ridge(alpha=ridge_s2.alpha_)
    else:
        final_s2 = LinearRegression()
    final_s2.fit(X_all_s2, y)

    predict_s2 = predict_df[active_features].copy()
    predict_s2[s2_other] = scaler_s2.transform(predict_s2[s2_other])
    s2_preds = final_s2.predict(predict_s2)

    print(f"Predictions: min={s2_preds.min():.1f}  mean={s2_preds.mean():.1f}  max={s2_preds.max():.1f}")

    from datetime import datetime
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"submission_{best_s2_name}_{ts}.csv"
    pd.DataFrame({"ID": predict_df["ID"], "W": s2_preds.round().astype(int)}).to_csv(filename, index=False)
    print(f"Saved: {filename}")

    msg = f"{best_s2_name} {len(active_features)}feat testMAE={best_s2_mae:.4f}"
    cmd = f'kaggle competitions submit -c {COMPETITION_NAME} -f "{filename}" -m "{msg}"'
    print(f"Submitting: {cmd}")
    os.system(cmd)
else:
    print(f"{best_s2_name} test MAE ({best_s2_mae:.4f}) >= 2.81 — skipping submission.")
    print("ElasticNet remains the best submitted model.")


TwoStage_Ridge test MAE (2.8029) < 2.81 — generating submission.
Predictions: min=44.9  mean=79.0  max=109.8
Saved: submission_TwoStage_Ridge_20260329_185739.csv
Submitting: kaggle competitions submit -c sctpdsai-m-3-ds-2-f-coaching-money-ball-analytics -f "submission_TwoStage_Ridge_20260329_185739.csv" -m "TwoStage_Ridge 36feat testMAE=2.8029"


100%|██████████| 3.34k/3.34k [00:00<00:00, 4.30kB/s]


Successfully submitted to SCTPDSAI-M3-DS2F-Coaching-MoneyBall Analytics

In [37]:
# --- Fine-grained ElasticNet grid around winning hyperparameters ---
# Original winner: alpha=0.005356, l1_ratio=0.99

from sklearn.linear_model import ElasticNetCV, ElasticNet

# Fine l1_ratio: probe 0.90–1.0 in small steps (1.0 = pure Lasso)
l1_fine = [0.90, 0.92, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 0.995, 1.0]

# Fine alpha: dense grid around 0.005356
alphas_fine = np.concatenate([
    np.logspace(-5, -3, 30),   # 0.00001 – 0.001
    np.linspace(0.001, 0.02, 40),  # 0.001 – 0.02 (neighbourhood of 0.005356)
    np.logspace(-1.5, 1, 20),  # 0.03 – 10
])
alphas_fine = np.sort(np.unique(alphas_fine))

print(f"Fine grid: {len(l1_fine)} l1_ratios x {len(alphas_fine)} alphas = {len(l1_fine)*len(alphas_fine)} combos")

kf_fine = KFold(n_splits=5, shuffle=True, random_state=42)

enet_fine = ElasticNetCV(
    l1_ratio=l1_fine,
    alphas=alphas_fine,
    cv=kf_fine,
    max_iter=100000,
    tol=1e-5,
    n_jobs=-1,
)
enet_fine.fit(X_train_scaled, y_train)

print(f"\nBest alpha:    {enet_fine.alpha_:.8f}")
print(f"Best l1_ratio: {enet_fine.l1_ratio_:.4f}")

# Compare with original
print(f"\nOriginal: alpha=0.005356, l1_ratio=0.99")
print(f"Fine:     alpha={enet_fine.alpha_:.8f}, l1_ratio={enet_fine.l1_ratio_}")

en_fine_train_preds = enet_fine.predict(X_train_scaled)
en_fine_test_preds  = enet_fine.predict(X_test_scaled)
en_fine_mae = mean_absolute_error(y_test, en_fine_test_preds)

print(f"\n=== Fine ElasticNet (80/20 split) ===")
print(f"  Train MAE:  {mean_absolute_error(y_train, en_fine_train_preds):.4f}")
print(f"  Test  MAE:  {en_fine_mae:.4f}")
print(f"  Test  RMSE: {mean_squared_error(y_test, en_fine_test_preds)**0.5:.4f}")
print(f"  Test  R2:   {r2_score(y_test, en_fine_test_preds):.4f}")

n_zero_fine = (enet_fine.coef_ == 0).sum()
print(f"\nFeature selection: {n_zero_fine}/{len(enet_fine.coef_)} zeroed out, {len(enet_fine.coef_)-n_zero_fine} active")

# 10-fold CV
kf_full = KFold(n_splits=10, shuffle=True, random_state=0)
enet_fine_full = ElasticNet(alpha=enet_fine.alpha_, l1_ratio=enet_fine.l1_ratio_, max_iter=100000, tol=1e-5)
en_fine_cv = cross_val_score(enet_fine_full, X_all_scaled, y, cv=kf_full, scoring='neg_mean_absolute_error')
print(f"\n10-fold CV MAE: {-en_fine_cv.mean():.4f} +/- {en_fine_cv.std():.4f}")

# Summary
print(f"\n{'='*55}")
print(f"  Model                    Test MAE   CV MAE")
print(f"  ElasticNet (original)    {en_test_mae:.4f}     {en_cv_mae:.4f}")
print(f"  ElasticNet (fine grid)   {en_fine_mae:.4f}     {-en_fine_cv.mean():.4f}")
print(f"  Improvement:             {en_test_mae - en_fine_mae:+.4f}     {en_cv_mae - (-en_fine_cv.mean()):+.4f}")
print(f"{'='*55}")


Fine grid: 10 l1_ratios x 89 alphas = 890 combos


/home/fredc/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.726e+00, tolerance: 2.041e+00
  model = cd_fast.enet_coordinate_descent_gram(
/home/fredc/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.314e+00, tolerance: 2.041e+00
  model = cd_fast.enet_coordinate_descent_gram(
/home/fredc/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or conside


Best alpha:    0.00733333
Best l1_ratio: 1.0000

Original: alpha=0.005356, l1_ratio=0.99
Fine:     alpha=0.00733333, l1_ratio=1.0

=== Fine ElasticNet (80/20 split) ===
  Train MAE:  2.6255
  Test  MAE:  2.8021
  Test  RMSE: 3.5261
  Test  R2:   0.9226

Feature selection: 33/68 zeroed out, 35 active

10-fold CV MAE: 2.7054 +/- 0.1596

  Model                    Test MAE   CV MAE
  ElasticNet (original)    2.8015     2.7055
  ElasticNet (fine grid)   2.8021     2.7054
  Improvement:             -0.0006     +0.0001


In [38]:
# --- Conditional submission: deploy fine ElasticNet if test MAE < 2.81 ---
if en_fine_mae < 2.81:
    print(f"Fine ElasticNet test MAE ({en_fine_mae:.4f}) < 2.81 — generating submission.")

    scaler_fine = StandardScaler()
    X_all_fine = X.copy()
    X_all_fine[other_cols] = scaler_fine.fit_transform(X[other_cols])

    final_fine = ElasticNet(
        alpha=enet_fine.alpha_,
        l1_ratio=enet_fine.l1_ratio_,
        max_iter=100000,
        tol=1e-5,
    )
    final_fine.fit(X_all_fine, y)

    predict_fine = predict_df[available_features].copy()
    predict_fine[other_cols] = scaler_fine.transform(predict_fine[other_cols])
    fine_preds = final_fine.predict(predict_fine)

    print(f"Predictions: min={fine_preds.min():.1f}  mean={fine_preds.mean():.1f}  max={fine_preds.max():.1f}")

    from datetime import datetime
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"submission_ElasticNet_fine_{ts}.csv"
    pd.DataFrame({"ID": predict_df["ID"], "W": fine_preds.round().astype(int)}).to_csv(filename, index=False)
    print(f"Saved: {filename}")

    msg = f"ElasticNet_fine alpha={enet_fine.alpha_:.8f} l1={enet_fine.l1_ratio_} testMAE={en_fine_mae:.4f}"
    cmd = f'kaggle competitions submit -c {COMPETITION_NAME} -f "{filename}" -m "{msg}"'
    print(f"Submitting: {cmd}")
    os.system(cmd)
else:
    print(f"Fine ElasticNet test MAE ({en_fine_mae:.4f}) >= 2.81 — skipping submission.")
    print("Original ElasticNet remains the best submitted model.")


Fine ElasticNet test MAE (2.8021) < 2.81 — generating submission.
Predictions: min=44.9  mean=79.0  max=109.6
Saved: submission_ElasticNet_fine_20260329_190226.csv
Submitting: kaggle competitions submit -c sctpdsai-m-3-ds-2-f-coaching-money-ball-analytics -f "submission_ElasticNet_fine_20260329_190226.csv" -m "ElasticNet_fine alpha=0.00733333 l1=1.0 testMAE=2.8021"


100%|██████████| 3.34k/3.34k [00:00<00:00, 3.86kB/s]


Successfully submitted to SCTPDSAI-M3-DS2F-Coaching-MoneyBall Analytics

In [40]:
from sklearn.linear_model import QuantileRegressor
from sklearn.model_selection import GridSearchCV, KFold

# --- Quantile Regression (median, alpha=0.5) ---
# Directly optimises MAE instead of MSE
# Uses L1 penalty (alpha param) — complements ElasticNet's sparsity finding

# Test on both feature sets
qr_params = {
    'alpha': [0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0],
}

kf_qr = KFold(n_splits=5, shuffle=True, random_state=42)

# Variant 1: QR on ElasticNet-selected features
X_train_active = X_train_scaled[active_features]
X_test_active  = X_test_scaled[active_features]

qr_sel = GridSearchCV(
    QuantileRegressor(quantile=0.5, solver='highs'),
    qr_params,
    cv=kf_qr,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
qr_sel.fit(X_train_active, y_train)

qr_sel_test_preds = qr_sel.predict(X_test_active)
qr_sel_mae = mean_absolute_error(y_test, qr_sel_test_preds)

print(f"=== Quantile Regression (selected {len(active_features)} features) ===")
print(f"  Best alpha:  {qr_sel.best_params_['alpha']}")
print(f"  Train MAE:   {mean_absolute_error(y_train, qr_sel.predict(X_train_active)):.4f}")
print(f"  Test  MAE:   {qr_sel_mae:.4f}")
print(f"  Test  RMSE:  {mean_squared_error(y_test, qr_sel_test_preds)**0.5:.4f}")
print(f"  Test  R2:    {r2_score(y_test, qr_sel_test_preds):.4f}")

# Variant 2: QR on all features
qr_all = GridSearchCV(
    QuantileRegressor(quantile=0.5, solver='highs'),
    qr_params,
    cv=kf_qr,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
qr_all.fit(X_train_scaled, y_train)

qr_all_test_preds = qr_all.predict(X_test_scaled)
qr_all_mae = mean_absolute_error(y_test, qr_all_test_preds)

print(f"\n=== Quantile Regression (all {len(available_features)} features) ===")
print(f"  Best alpha:  {qr_all.best_params_['alpha']}")
print(f"  Train MAE:   {mean_absolute_error(y_train, qr_all.predict(X_train_scaled)):.4f}")
print(f"  Test  MAE:   {qr_all_mae:.4f}")
print(f"  Test  RMSE:  {mean_squared_error(y_test, qr_all_test_preds)**0.5:.4f}")
print(f"  Test  R2:    {r2_score(y_test, qr_all_test_preds):.4f}")

# 10-fold CV on full data
X_all_active = X_all_scaled[active_features]
kf_full = KFold(n_splits=10, shuffle=True, random_state=0)

qr_sel_cv = cross_val_score(
    QuantileRegressor(quantile=0.5, alpha=qr_sel.best_params_['alpha'], solver='highs'),
    X_all_active, y, cv=kf_full, scoring='neg_mean_absolute_error'
)
qr_all_cv = cross_val_score(
    QuantileRegressor(quantile=0.5, alpha=qr_all.best_params_['alpha'], solver='highs'),
    X_all_scaled, y, cv=kf_full, scoring='neg_mean_absolute_error'
)

print(f"\n=== 10-fold CV (full data) ===")
print(f"  QR (selected) CV MAE: {-qr_sel_cv.mean():.4f} +/- {qr_sel_cv.std():.4f}")
print(f"  QR (all feat) CV MAE: {-qr_all_cv.mean():.4f} +/- {qr_all_cv.std():.4f}")

# Summary
print(f"\n{'='*55}")
print(f"  Model                    Test MAE   CV MAE")
print(f"  ElasticNet (best)        {en_test_mae:.4f}     {en_cv_mae:.4f}")
print(f"  QR (selected feat)       {qr_sel_mae:.4f}     {-qr_sel_cv.mean():.4f}")
print(f"  QR (all feat)            {qr_all_mae:.4f}     {-qr_all_cv.mean():.4f}")
print(f"{'='*55}")

# Pick best QR variant
best_qr_name, best_qr_mae, best_qr_model, best_qr_features = (
    ("QR_all", qr_all_mae, qr_all, available_features)
    if qr_all_mae <= qr_sel_mae
    else ("QR_selected", qr_sel_mae, qr_sel, active_features)
)
print(f"\nBest QR variant: {best_qr_name} (test MAE={best_qr_mae:.4f})")


=== Quantile Regression (selected 36 features) ===
  Best alpha:  0.001
  Train MAE:   2.6045
  Test  MAE:   2.8064
  Test  RMSE:  3.5290
  Test  R2:    0.9224

=== Quantile Regression (all 68 features) ===
  Best alpha:  0.001
  Train MAE:   2.6059
  Test  MAE:   2.8071
  Test  RMSE:  3.5315
  Test  R2:    0.9223

=== 10-fold CV (full data) ===
  QR (selected) CV MAE: 2.7104 +/- 0.1541
  QR (all feat) CV MAE: 2.7265 +/- 0.1571

  Model                    Test MAE   CV MAE
  ElasticNet (best)        2.8015     2.7055
  QR (selected feat)       2.8064     2.7104
  QR (all feat)            2.8071     2.7265

Best QR variant: QR_selected (test MAE=2.8064)


In [41]:
# --- Conditional submission: deploy best QR if test MAE < 2.81 ---
if best_qr_mae < 2.81:
    print(f"{best_qr_name} test MAE ({best_qr_mae:.4f}) < 2.81 — generating submission.")

    best_qr_params = best_qr_model.best_params_
    qr_other = [c for c in best_qr_features if c not in one_hot_cols]

    scaler_qr = StandardScaler()
    X_all_qr = X[best_qr_features].copy()
    X_all_qr[qr_other] = scaler_qr.fit_transform(X_all_qr[qr_other])

    final_qr = QuantileRegressor(
        quantile=0.5,
        alpha=best_qr_params['alpha'],
        solver='highs',
    )
    final_qr.fit(X_all_qr, y)

    predict_qr = predict_df[best_qr_features].copy()
    predict_qr[qr_other] = scaler_qr.transform(predict_qr[qr_other])
    qr_preds = final_qr.predict(predict_qr)

    print(f"Predictions: min={qr_preds.min():.1f}  mean={qr_preds.mean():.1f}  max={qr_preds.max():.1f}")

    from datetime import datetime
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"submission_{best_qr_name}_{ts}.csv"
    pd.DataFrame({"ID": predict_df["ID"], "W": qr_preds.round().astype(int)}).to_csv(filename, index=False)
    print(f"Saved: {filename}")

    msg = f"{best_qr_name} alpha={best_qr_params['alpha']} testMAE={best_qr_mae:.4f}"
    cmd = f'kaggle competitions submit -c {COMPETITION_NAME} -f "{filename}" -m "{msg}"'
    print(f"Submitting: {cmd}")
    os.system(cmd)
else:
    print(f"{best_qr_name} test MAE ({best_qr_mae:.4f}) >= 2.81 — skipping submission.")
    print("ElasticNet remains the best submitted model.")


QR_selected test MAE (2.8064) < 2.81 — generating submission.
Predictions: min=45.7  mean=79.0  max=109.5
Saved: submission_QR_selected_20260329_191528.csv
Submitting: kaggle competitions submit -c sctpdsai-m-3-ds-2-f-coaching-money-ball-analytics -f "submission_QR_selected_20260329_191528.csv" -m "QR_selected alpha=0.001 testMAE=2.8064"


100%|██████████| 3.34k/3.34k [00:00<00:00, 4.15kB/s]


Successfully submitted to SCTPDSAI-M3-DS2F-Coaching-MoneyBall Analytics

In [ ]:
# --- Prediction Averaging: QR + ElasticNet ---
# Both models are well-calibrated linear models with different loss functions.
# Averaging cancels independent errors without overfitting risk.

import glob

# Load the most recent QR and ElasticNet submission files
qr_files = sorted(glob.glob('submission_QR_*.csv'))
en_files = sorted(glob.glob('submission_ElasticNet_2*.csv'))  # exclude "fine" variant

print(f"QR submission:         {qr_files[-1] if qr_files else 'NOT FOUND'}")
print(f"ElasticNet submission: {en_files[-1] if en_files else 'NOT FOUND'}")

qr_sub = pd.read_csv(qr_files[-1])
en_sub = pd.read_csv(en_files[-1])

# Verify IDs match
assert (qr_sub['ID'] == en_sub['ID']).all(), "ID mismatch!"

# Simple average
avg_preds = (qr_sub['W'] + en_sub['W']) / 2

# Also try weighted averages tilted toward QR (the better model)
weights_to_try = [
    ("50-50", 0.5, 0.5),
    ("60-40 QR", 0.6, 0.4),
    ("70-30 QR", 0.7, 0.3),
]

print(f"\nPrediction comparison:")
print(f"  QR:         mean={qr_sub['W'].mean():.2f}  std={qr_sub['W'].std():.2f}")
print(f"  ElasticNet: mean={en_sub['W'].mean():.2f}  std={en_sub['W'].std():.2f}")
print(f"  Avg (50/50):mean={avg_preds.mean():.2f}  std={avg_preds.std():.2f}")
print(f"\n  Prediction disagreement (MAD): {(qr_sub['W'] - en_sub['W']).abs().mean():.2f}")

from datetime import datetime
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

for name, w_qr, w_en in weights_to_try:
    blended = (w_qr * qr_sub['W'] + w_en * en_sub['W']).round().astype(int)
    fname = f"submission_avg_{name.replace(' ','_')}_{ts}.csv"
    pd.DataFrame({"ID": qr_sub["ID"], "W": blended}).to_csv(fname, index=False)
    print(f"\nSaved: {fname}")

# Submit the 50-50 average (safest bet)
submit_file = f"submission_avg_50-50_{ts}.csv"
msg = f"Avg QR+ElasticNet 50-50"
cmd = f'kaggle competitions submit -c {COMPETITION_NAME} -f "{submit_file}" -m "{msg}"'
print(f"\nSubmitting: {cmd}")
os.system(cmd)

# Submit the 60-40 QR-weighted average
submit_file_60 = f"submission_avg_60-40_QR_{ts}.csv"
msg_60 = f"Avg QR+ElasticNet 60-40"
cmd_60 = f'kaggle competitions submit -c {COMPETITION_NAME} -f "{submit_file_60}" -m "{msg_60}"'
print(f"\nSubmitting: {cmd_60}")
_ = os.system(cmd_60)


In [44]:
# --- Three-Model Average: QR + ElasticNet + Ridge ---
import glob

qr_files = sorted(glob.glob('submission_QR_*.csv'))
en_files = sorted(glob.glob('submission_ElasticNet_2*.csv'))

qr_sub = pd.read_csv(qr_files[-1])
en_sub = pd.read_csv(en_files[-1])
ridge_sub = pd.read_csv('submission_predict.csv')  # Ridge submission

assert (qr_sub['ID'] == en_sub['ID']).all()
assert (qr_sub['ID'] == ridge_sub['ID']).all()

print(f"QR:         mean={qr_sub['W'].mean():.2f}  std={qr_sub['W'].std():.2f}")
print(f"ElasticNet: mean={en_sub['W'].mean():.2f}  std={en_sub['W'].std():.2f}")
print(f"Ridge:      mean={ridge_sub['W'].mean():.2f}  std={ridge_sub['W'].std():.2f}")

# Pairwise disagreement
print(f"\nPairwise MAD:")
print(f"  QR vs EN:    {(qr_sub['W'] - en_sub['W']).abs().mean():.2f}")
print(f"  QR vs Ridge: {(qr_sub['W'] - ridge_sub['W']).abs().mean():.2f}")
print(f"  EN vs Ridge: {(en_sub['W'] - ridge_sub['W']).abs().mean():.2f}")

from datetime import datetime
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# Equal-weight three-way average
avg3 = ((qr_sub['W'] + en_sub['W'] + ridge_sub['W']) / 3).round().astype(int)
fname3 = f"submission_avg3_equal_{ts}.csv"
pd.DataFrame({"ID": qr_sub["ID"], "W": avg3}).to_csv(fname3, index=False)
print(f"\nSaved: {fname3}")
print(f"  mean={avg3.mean():.2f}  std={avg3.std():.2f}")

msg = "Avg3 QR+ElasticNet+Ridge equal"
cmd = f'kaggle competitions submit -c {COMPETITION_NAME} -f "{fname3}" -m "{msg}"'
print(f"\nSubmitting: {cmd}")
_ = os.system(cmd)


QR:         mean=79.03  std=12.04
ElasticNet: mean=79.04  std=12.08
Ridge:      mean=79.04  std=12.11

Pairwise MAD:
  QR vs EN:    0.23
  QR vs Ridge: 0.29
  EN vs Ridge: 0.14

Saved: submission_avg3_equal_20260329_192614.csv
  mean=79.04  std=12.10

Submitting: kaggle competitions submit -c sctpdsai-m-3-ds-2-f-coaching-money-ball-analytics -f "submission_avg3_equal_20260329_192614.csv" -m "Avg3 QR+ElasticNet+Ridge equal"


100%|██████████| 3.34k/3.34k [00:00<00:00, 3.97kB/s]


Successfully submitted to SCTPDSAI-M3-DS2F-Coaching-MoneyBall Analytics

In [45]:
from sklearn.linear_model import Ridge

# Refit scaler + model on ALL training data
scaler_final = StandardScaler()
X_all_final  = X.copy()
X_all_final[other_cols] = scaler_final.fit_transform(X[other_cols])

final_model = Ridge(alpha=ridge.alpha_)
final_model.fit(X_all_final, y)

predict_features = predict_df[available_features].copy()
predict_features[other_cols] = scaler_final.transform(predict_features[other_cols])
predict_preds = final_model.predict(predict_features)

print(f"Predictions: min={predict_preds.min():.1f}  mean={predict_preds.mean():.1f}  max={predict_preds.max():.1f}")

import pandas as pd
submission_df = pd.DataFrame({'ID': predict_df['ID'], 'W': predict_preds.round().astype(int)})
submission_df.to_csv('submission_predict.csv', index=False)
print("Saved: submission_predict.csv")
print(submission_df.head().to_string())

Predictions: min=44.6  mean=79.1  max=110.0
Saved: submission_predict.csv
     ID   W
0  1756  70
1  1282  75
2   351  85
3   421  87
4    57  93


In [10]:
# =============================================================
# Target Transformation: Predict Win% (W/G) then multiply by G
# Normalises across season lengths and eras
# =============================================================
from sklearn.linear_model import ElasticNetCV, ElasticNet, RidgeCV, Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os

# --- Create win-percentage target ---
y_wpct = data_df['W'] / data_df['G']   # target: win fraction [0, 1]
G_train = X_train['G'].values
G_test  = X_test['G'].values

y_wpct_train = y_train / G_train
y_wpct_test  = y_test / G_test

print(f"Win% target — train mean: {y_wpct_train.mean():.4f}, std: {y_wpct_train.std():.4f}")
print(f"Win% target — test  mean: {y_wpct_test.mean():.4f}, std: {y_wpct_test.std():.4f}")

# --- 1) ElasticNet on win% ---
l1_ratios_wp = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
alphas_wp    = np.logspace(-6, 0, 80)

kf_wp = KFold(n_splits=5, shuffle=True, random_state=42)
enet_wp = ElasticNetCV(
    l1_ratio=l1_ratios_wp,
    alphas=alphas_wp,
    cv=kf_wp,
    max_iter=50000,
    tol=1e-5,
    n_jobs=-1,
)
enet_wp.fit(X_train_scaled, y_wpct_train)

print(f"\nElasticNet (win%) — alpha: {enet_wp.alpha_:.6f}, l1_ratio: {enet_wp.l1_ratio_:.4f}")

# Predict win%, convert back to wins
wp_train_preds = enet_wp.predict(X_train_scaled) * G_train
wp_test_preds  = enet_wp.predict(X_test_scaled)  * G_test

wp_en_mae  = mean_absolute_error(y_test, wp_test_preds)
wp_en_rmse = mean_squared_error(y_test, wp_test_preds) ** 0.5
wp_en_r2   = r2_score(y_test, wp_test_preds)

print(f"  Train MAE (W): {mean_absolute_error(y_train, wp_train_preds):.4f}")
print(f"  Test  MAE (W): {wp_en_mae:.4f}")
print(f"  Test  RMSE:    {wp_en_rmse:.4f}")
print(f"  Test  R²:      {wp_en_r2:.4f}")

n_active = (enet_wp.coef_ != 0).sum()
print(f"  Active features: {n_active}/{len(enet_wp.coef_)}")

# --- 2) RidgeCV on win% ---
alphas_r = np.logspace(-3, 5, 100)
ridge_wp = RidgeCV(alphas=alphas_r, cv=5, scoring='neg_mean_absolute_error')
ridge_wp.fit(X_train_scaled, y_wpct_train)

rw_test_preds = ridge_wp.predict(X_test_scaled) * G_test
wp_ridge_mae  = mean_absolute_error(y_test, rw_test_preds)

print(f"\nRidge (win%) — alpha: {ridge_wp.alpha_:.4f}")
print(f"  Test MAE (W): {wp_ridge_mae:.4f}")

# --- 10-fold CV on full data (best of the two) ---
best_wp_name, best_wp_mae = ("ElasticNet", wp_en_mae) if wp_en_mae < wp_ridge_mae else ("Ridge", wp_ridge_mae)
print(f"\nBest win% model: {best_wp_name} (test MAE {best_wp_mae:.4f})")

if best_wp_name == "ElasticNet":
    cv_model_wp = ElasticNet(alpha=enet_wp.alpha_, l1_ratio=enet_wp.l1_ratio_, max_iter=50000)
else:
    cv_model_wp = Ridge(alpha=ridge_wp.alpha_)

kf_full = KFold(n_splits=10, shuffle=True, random_state=0)
G_all = X['G'].values

# Manual CV to do the W/G → W conversion properly
from sklearn.base import clone
cv_maes = []
for train_idx, val_idx in kf_full.split(X_all_scaled):
    m = clone(cv_model_wp)
    m.fit(X_all_scaled.iloc[train_idx], y_wpct.iloc[train_idx])
    val_preds_w = m.predict(X_all_scaled.iloc[val_idx]) * G_all[val_idx]
    cv_maes.append(mean_absolute_error(y.iloc[val_idx], val_preds_w))

cv_mae_wp = np.mean(cv_maes)
print(f"10-fold CV MAE (W, full data): {cv_mae_wp:.4f} ± {np.std(cv_maes):.4f}")

# --- Comparison with direct-W models ---
print(f"\n--- Comparison (test MAE) ---")
print(f"  Direct W ElasticNet: {en_test_mae:.4f}")
print(f"  Win% → W ElasticNet: {wp_en_mae:.4f}  ({en_test_mae - wp_en_mae:+.4f})")
print(f"  Win% → W Ridge:      {wp_ridge_mae:.4f}")

# --- Conditional submission ---
COMPETITION_NAME = 'sctpdsai-m-3-ds-2-f-coaching-money-ball-analytics'

if best_wp_mae < 2.81:
    print(f"\n{best_wp_name} win% test MAE ({best_wp_mae:.4f}) < 2.81 — generating submission.")

    scaler_wp = StandardScaler()
    X_all_wp  = X.copy()
    X_all_wp[other_cols] = scaler_wp.fit_transform(X[other_cols])

    final_wp = clone(cv_model_wp)
    final_wp.fit(X_all_wp, y_wpct)

    predict_features_wp = predict_df[available_features].copy()
    predict_features_wp[other_cols] = scaler_wp.transform(predict_features_wp[other_cols])
    G_pred = predict_df['G'].values

    wp_preds = final_wp.predict(predict_features_wp) * G_pred

    print(f"Predictions: min={wp_preds.min():.1f}  mean={wp_preds.mean():.1f}  max={wp_preds.max():.1f}")

    from datetime import datetime
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"submission_WinPct_{best_wp_name}_{ts}.csv"
    pd.DataFrame({"ID": predict_df["ID"], "W": wp_preds.round().astype(int)}).to_csv(filename, index=False)
    print(f"Saved: {filename}")

    msg = f"WinPct_{best_wp_name} testMAE={best_wp_mae:.4f} cvMAE={cv_mae_wp:.4f}"
    cmd = f'kaggle competitions submit -c {COMPETITION_NAME} -f "{filename}" -m "{msg}"'
    print(f"Submitting: {cmd}")
    os.system(cmd)
else:
    print(f"\nBest win% model MAE ({best_wp_mae:.4f}) >= 2.81 — skipping submission.")

Win% target — train mean: 0.4983, std: 0.0826
Win% target — test  mean: 0.4981, std: 0.0792


/home/fredc/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.715e-04, tolerance: 7.913e-05
  model = cd_fast.enet_coordinate_descent_gram(
/home/fredc/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.815e-04, tolerance: 7.918e-05
  model = cd_fast.enet_coordinate_descent_gram(
/home/fredc/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or conside


ElasticNet (win%) — alpha: 0.000056, l1_ratio: 0.9500
  Train MAE (W): 2.6290
  Test  MAE (W): 2.8178
  Test  RMSE:    3.5400
  Test  R²:      0.9220
  Active features: 35/68

Ridge (win%) — alpha: 1.1768
  Test MAE (W): 2.8108

Best win% model: Ridge (test MAE 2.8108)
10-fold CV MAE (W, full data): 2.7101 ± 0.1556

--- Comparison (test MAE) ---
  Direct W ElasticNet: 2.8015
  Win% → W ElasticNet: 2.8178  (-0.0163)
  Win% → W Ridge:      2.8108

Best win% model MAE (2.8108) >= 2.81 — skipping submission.
